# Weather × AIS Exploration

This notebook investigates how weather conditions can enrich the AIS dataset to improve delay prediction.

Weather affects cargo vessels in two distinct ways:

1. **Vessel-position conditions** — waves, swell, and wind along the route slow the vessel down or force speed reductions for safety (heavy weather routing). These affect speed-over-ground directly.
2. **Destination-port conditions** — strong winds, low visibility, and precipitation at the port can prevent berthing, extend pilotage windows, or trigger port closures. These introduce terminal delays that are invisible in the kinematic features.

**Data source:** [Open-Meteo](https://open-meteo.com/) — free, no API key required, backed by ERA5 / ECMWF reanalysis for historical dates.

- Marine API (`marine-api.open-meteo.com`) → wave height, swell, ocean current
- Archive API (`archive-api.open-meteo.com`) → wind speed/direction, precipitation, weather codes

**Scope:** the same nine-day window used in `training.ipynb` (2025-01-01 to 2025-01-09), US cargo vessels.

In [ ]:
import sys
import time
from pathlib import Path

sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import requests
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from shapely.geometry import Point

from src.methods import dms_to_dd

pd.set_option('display.max_columns', 30)
%matplotlib inline

## 1. Load AIS voyage data

Re-use the labeled pings and voyage records cached by `training.ipynb`.
Run that notebook first if `data/cache/` does not exist.

In [ ]:
cache_dir = Path('../data/cache')
labeled_path = cache_dir / 'labeled.parquet'
voyages_path = cache_dir / 'voyages.parquet'

if not labeled_path.exists():
    raise FileNotFoundError(
        'Cache not found. Run training.ipynb first to generate data/cache/labeled.parquet '
        'and data/cache/voyages.parquet.'
    )

df_labeled = pq.read_table(labeled_path, use_pandas_metadata=False).to_pandas()
df_labeled['base_date_time'] = pd.to_datetime(df_labeled['base_date_time'])

df_voyages = pq.read_table(voyages_path, use_pandas_metadata=False).to_pandas()
df_voyages['departure_time'] = pd.to_datetime(df_voyages['departure_time'])
df_voyages['arrival_time'] = pd.to_datetime(df_voyages['arrival_time'])

# Port coordinates lookup
df_ports = pd.read_csv('../data/ports/ports.csv')
df_ports['lat_dd'] = df_ports['latitude'].apply(dms_to_dd)
df_ports['lon_dd'] = df_ports['longitude'].apply(dms_to_dd)
port_loc = df_ports.set_index('portName')[['lat_dd', 'lon_dd']]

print(f'Labeled pings : {len(df_labeled):,}')
print(f'Voyages       : {len(df_voyages):,}')
print(f'Date range    : {df_labeled["base_date_time"].min().date()} → {df_labeled["base_date_time"].max().date()}')

In [ ]:
# Narrow to sea pings inside a detected voyage (origin + destination known)
df_sea = df_labeled[
    df_labeled['voyage_id'].notna() & df_labeled['destination_port'].notna()
].copy()
df_sea['voyage_id'] = df_sea['voyage_id'].astype(int)

# Join arrival time and destination coordinates
df_sea = df_sea.merge(
    df_voyages[['voyage_id', 'arrival_time']],
    on='voyage_id', how='left',
)
df_sea = df_sea.join(
    port_loc.rename(columns={'lat_dd': 'dest_lat', 'lon_dd': 'dest_lon'}),
    on='destination_port',
)

# Remaining travel time label
df_sea['remaining_travel_hours'] = (
    (df_sea['arrival_time'] - df_sea['base_date_time']).dt.total_seconds() / 3600
)
df_sea = df_sea[df_sea['remaining_travel_hours'] >= 0]

print(f'Sea pings  : {len(df_sea):,}')
print(f'Voyages    : {df_sea["voyage_id"].nunique():,}')
print(f'Vessels    : {df_sea["mmsi"].nunique():,}')
print()
display(df_sea[['mmsi', 'base_date_time', 'latitude', 'longitude', 'cog', 'sog',
                'origin_port', 'destination_port', 'remaining_travel_hours']].head())

## 2. Fetching strategy

AIS pings arrive every 1–6 minutes, but meteorological reanalysis data is **hourly**.
Making one API call per ping would be wasteful and hit rate limits.

### Efficient approach

1. **Round** vessel positions to a 1° grid (~60 nm resolution — fine enough for weather, coarse enough to collapse many pings into the same grid cell).
2. **Identify** unique `(lat_bin, lon_bin)` cells touched by any ping.
3. **One API call per grid cell** returning the full hourly time series for the entire date range.
4. **Floor** each AIS timestamp to the hour and **join** on `(lat_bin, lon_bin, hour)`.

This typically reduces API calls by two orders of magnitude relative to per-ping fetching.
For destination ports the approach is even simpler: one call per unique port.

All responses are cached to `data/cache/weather/` so subsequent runs are instant.

In [ ]:
WEATHER_CACHE = Path('../data/cache/weather')
WEATHER_CACHE.mkdir(parents=True, exist_ok=True)

DATE_START = df_sea['base_date_time'].min().date().isoformat()
DATE_END   = df_sea['base_date_time'].max().date().isoformat()

print(f'Fetching weather for {DATE_START} → {DATE_END}')


def _get(url: str, params: dict, retries: int = 3) -> dict:
    """GET request with exponential back-off retry."""
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r.json()
        except requests.RequestException as exc:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)


def fetch_marine_weather(lat: float, lon: float,
                         start: str, end: str) -> pd.DataFrame:
    """Hourly marine weather from Open-Meteo Marine API.

    Returns a DataFrame indexed by UTC hour with columns:
        wave_height (m), wave_direction (°), wave_period (s),
        swell_wave_height (m), ocean_current_velocity (m/s),
        ocean_current_direction (°)

    Note: returns NaN for inland / river locations not covered by
    the ocean model.
    """
    data = _get(
        'https://marine-api.open-meteo.com/v1/marine',
        params=dict(
            latitude=lat, longitude=lon,
            hourly=','.join([
                'wave_height', 'wave_direction', 'wave_period',
                'swell_wave_height', 'ocean_current_velocity',
                'ocean_current_direction',
            ]),
            start_date=start, end_date=end,
            timezone='UTC',
        ),
    )
    h = data['hourly']
    return pd.DataFrame({
        'time': pd.to_datetime(h['time']),
        'wave_height': h['wave_height'],
        'wave_direction': h['wave_direction'],
        'wave_period': h['wave_period'],
        'swell_wave_height': h['swell_wave_height'],
        'ocean_current_velocity': h['ocean_current_velocity'],
        'ocean_current_direction': h['ocean_current_direction'],
    })


def fetch_weather(lat: float, lon: float,
                  start: str, end: str) -> pd.DataFrame:
    """Hourly surface weather from Open-Meteo Archive API (ERA5).

    Returns a DataFrame indexed by UTC hour with columns:
        wind_speed_10m (m/s), wind_gusts_10m (m/s),
        wind_direction_10m (°), precipitation (mm),
        weather_code (WMO), visibility (m)
    """
    data = _get(
        'https://archive-api.open-meteo.com/v1/archive',
        params=dict(
            latitude=lat, longitude=lon,
            hourly=','.join([
                'wind_speed_10m', 'wind_gusts_10m', 'wind_direction_10m',
                'precipitation', 'weather_code', 'visibility',
            ]),
            wind_speed_unit='ms',
            start_date=start, end_date=end,
            timezone='UTC',
        ),
    )
    h = data['hourly']
    return pd.DataFrame({
        'time': pd.to_datetime(h['time']),
        'wind_speed_10m': h['wind_speed_10m'],
        'wind_gusts_10m': h['wind_gusts_10m'],
        'wind_direction_10m': h['wind_direction_10m'],
        'precipitation': h['precipitation'],
        'weather_code': h['weather_code'],
        'visibility': h.get('visibility'),
    })

## 3. Vessel-position weather

### 3a. Marine conditions (waves, swell, current)

The Open-Meteo Marine API provides ERA5-derived ocean model data. Coverage is limited to open-ocean and coastal areas; river / inland waterway positions will return `NaN` values, which we track explicitly.

In [ ]:
# Bin vessel positions to a 1° grid
df_sea['lat_bin'] = df_sea['latitude'].round(0)
df_sea['lon_bin'] = df_sea['longitude'].round(0)
df_sea['hour'] = df_sea['base_date_time'].dt.floor('h')

location_bins = df_sea[['lat_bin', 'lon_bin']].drop_duplicates().reset_index(drop=True)
print(f'Unique 1° location bins: {len(location_bins)}')
print(f'Covers bounding box: lat [{location_bins.lat_bin.min()}, {location_bins.lat_bin.max()}], '
      f'lon [{location_bins.lon_bin.min()}, {location_bins.lon_bin.max()}]')

In [ ]:
marine_dfs = []
skipped = 0

for _, row in location_bins.iterrows():
    lat, lon = row['lat_bin'], row['lon_bin']
    cache_file = WEATHER_CACHE / f'marine_{lat:.0f}_{lon:.0f}.parquet'

    if cache_file.exists():
        df_w = pd.read_parquet(cache_file)
    else:
        try:
            df_w = fetch_marine_weather(lat, lon, DATE_START, DATE_END)
            df_w['lat_bin'] = lat
            df_w['lon_bin'] = lon
            df_w.to_parquet(cache_file, index=False)
            time.sleep(0.15)  # polite rate limiting
        except Exception as exc:
            print(f'  Marine API failed for ({lat}, {lon}): {exc}')
            skipped += 1
            continue

    marine_dfs.append(df_w)

df_marine = pd.concat(marine_dfs, ignore_index=True)
df_marine['time'] = pd.to_datetime(df_marine['time'])

print(f'Marine weather records fetched : {len(df_marine):,}')
print(f'Location bins skipped          : {skipped}')
print()
display(df_marine.describe().round(2))

### 3b. Wind at vessel position

The archive (ERA5) weather API provides surface wind for all land and sea locations,
including rivers and ports. Wind is the primary weather factor for vessels in protected
waterways where wave data is unavailable.

In [ ]:
wind_dfs = []
skipped_wind = 0

for _, row in location_bins.iterrows():
    lat, lon = row['lat_bin'], row['lon_bin']
    cache_file = WEATHER_CACHE / f'wind_{lat:.0f}_{lon:.0f}.parquet'

    if cache_file.exists():
        df_w = pd.read_parquet(cache_file)
    else:
        try:
            df_w = fetch_weather(lat, lon, DATE_START, DATE_END)
            df_w['lat_bin'] = lat
            df_w['lon_bin'] = lon
            df_w.to_parquet(cache_file, index=False)
            time.sleep(0.15)
        except Exception as exc:
            print(f'  Wind API failed for ({lat}, {lon}): {exc}')
            skipped_wind += 1
            continue

    wind_dfs.append(df_w)

df_wind = pd.concat(wind_dfs, ignore_index=True)
df_wind['time'] = pd.to_datetime(df_wind['time'])

print(f'Wind weather records fetched : {len(df_wind):,}')
print(f'Location bins skipped        : {skipped_wind}')

In [ ]:
# Merge marine conditions onto sea pings via (lat_bin, lon_bin, hour)
df_sea = df_sea.merge(
    df_marine.rename(columns={'time': 'hour'}),
    on=['lat_bin', 'lon_bin', 'hour'],
    how='left',
)

# Merge wind conditions
df_sea = df_sea.merge(
    df_wind.rename(columns={'time': 'hour'}),
    on=['lat_bin', 'lon_bin', 'hour'],
    how='left',
)

# Coverage
marine_coverage = df_sea['wave_height'].notna().mean()
wind_coverage   = df_sea['wind_speed_10m'].notna().mean()

print(f'Marine wave data coverage : {marine_coverage:.1%}  '
      f'({df_sea["wave_height"].notna().sum():,} / {len(df_sea):,} pings)')
print(f'Wind data coverage        : {wind_coverage:.1%}  '
      f'({df_sea["wind_speed_10m"].notna().sum():,} / {len(df_sea):,} pings)')
print()
print('NaN rate by column:')
weather_cols = ['wave_height', 'wave_period', 'swell_wave_height',
                'ocean_current_velocity', 'wind_speed_10m', 'wind_gusts_10m',
                'precipitation', 'visibility']
display(df_sea[weather_cols].isna().mean().rename('null_rate').to_frame().style.format('{:.1%}'))

### 3c. Vessel-position weather — exploratory analysis

Examine the distributions of wave and wind conditions, then quantify their relationship with speed-over-ground.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Weather at vessel position — distribution across all sea pings', fontsize=13)

cols = [
    ('wave_height',            'Wave height (m)',              'steelblue'),
    ('wave_period',            'Wave period (s)',              'royalblue'),
    ('swell_wave_height',      'Swell wave height (m)',        'dodgerblue'),
    ('ocean_current_velocity', 'Ocean current velocity (m/s)', 'teal'),
    ('wind_speed_10m',         'Wind speed 10m (m/s)',         'darkorange'),
    ('wind_gusts_10m',         'Wind gusts 10m (m/s)',         'orangered'),
]

for ax, (col, label, colour) in zip(axes.flat, cols):
    series = df_sea[col].dropna()
    if series.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
    else:
        ax.hist(series, bins=40, color=colour, edgecolor='white', linewidth=0.3)
        ax.axvline(series.median(), color='black', linestyle='--', linewidth=1,
                   label=f'Median: {series.median():.2f}')
        ax.legend(fontsize=8)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel(label)
    ax.set_ylabel('Ping count')

plt.tight_layout()
plt.show()

In [ ]:
# Bin wave height and wind speed into quartiles, then look at median SOG per bin
df_valid = df_sea.dropna(subset=['wave_height', 'wind_speed_10m', 'sog']).copy()

df_valid['wave_bin'] = pd.qcut(df_valid['wave_height'],   q=5, labels=False, duplicates='drop')
df_valid['wind_bin'] = pd.qcut(df_valid['wind_speed_10m'], q=5, labels=False, duplicates='drop')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Median SOG by weather conditions at vessel position', fontsize=12)

for ax, (bin_col, value_col, xlabel, colour) in zip(axes, [
    ('wave_bin',  'wave_height',    'Significant wave height (m) quintile', 'steelblue'),
    ('wind_bin',  'wind_speed_10m', 'Wind speed 10m (m/s) quintile',        'darkorange'),
]):
    grp = (
        df_valid.groupby(bin_col)
        .agg(
            median_sog=('sog', 'median'),
            q1_sog=('sog', lambda x: x.quantile(0.25)),
            q3_sog=('sog', lambda x: x.quantile(0.75)),
            mid_value=(value_col, 'median'),
        )
        .reset_index()
    )
    ax.bar(range(len(grp)), grp['median_sog'], color=colour, alpha=0.7)
    ax.errorbar(
        range(len(grp)),
        grp['median_sog'],
        yerr=[grp['median_sog'] - grp['q1_sog'], grp['q3_sog'] - grp['median_sog']],
        fmt='none', color='black', capsize=4, linewidth=1.2,
    )
    ax.set_xticks(range(len(grp)))
    ax.set_xticklabels([f'Q{i+1}\n({v:.1f})' for i, v in enumerate(grp['mid_value'])], fontsize=8)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Median SOG (knots)')
    ax.set_title(xlabel.split('(')[0].strip())

plt.tight_layout()
plt.show()

In [ ]:
# Pick the voyage with the highest median wave height to visualise weather along its track
voyage_wave_median = (
    df_sea.dropna(subset=['wave_height'])
    .groupby('voyage_id')['wave_height'].median()
    .sort_values(ascending=False)
)

sample_voyage_id = voyage_wave_median.index[0]
df_sample = df_sea[df_sea['voyage_id'] == sample_voyage_id].sort_values('base_date_time')

origin = df_sample['origin_port'].iloc[0]
dest   = df_sample['destination_port'].iloc[0]
print(f'Sample voyage {sample_voyage_id}: {origin} → {dest}')
print(f'Duration: {df_sample["base_date_time"].min()} → {df_sample["base_date_time"].max()}')
print(f'Median wave height: {df_sample["wave_height"].median():.2f} m')

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
fig.suptitle(f'Voyage {sample_voyage_id}: {origin} → {dest}', fontsize=12)

t = df_sample['base_date_time']

axes[0].plot(t, df_sample['sog'], color='black', linewidth=0.8, label='SOG (knots)')
axes[0].set_ylabel('SOG (knots)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].fill_between(t, df_sample['wave_height'].fillna(0), alpha=0.5,
                     color='steelblue', label='Wave height (m)')
axes[1].fill_between(t, df_sample['swell_wave_height'].fillna(0), alpha=0.4,
                     color='royalblue', label='Swell height (m)')
axes[1].set_ylabel('Wave / Swell (m)')
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].fill_between(t, df_sample['wind_speed_10m'].fillna(0), alpha=0.5,
                     color='darkorange', label='Wind speed (m/s)')
axes[2].fill_between(t, df_sample['wind_gusts_10m'].fillna(0), alpha=0.25,
                     color='orangered', label='Wind gusts (m/s)')
axes[2].set_ylabel('Wind (m/s)')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Destination-port weather

Port conditions affect the **final stage** of a voyage. High wind speeds above port authority thresholds delay docking; precipitation and low visibility affect pilotage. These factors can add hours of waiting time that are completely invisible to kinematic features.

Strategy: one API call per unique destination port covering the full date range.

In [ ]:
# Unique destination ports with their coordinates
dest_ports = (
    df_sea[['destination_port', 'dest_lat', 'dest_lon']]
    .drop_duplicates('destination_port')
    .dropna(subset=['dest_lat', 'dest_lon'])
    .reset_index(drop=True)
)
print(f'Unique destination ports: {len(dest_ports)}')
display(dest_ports.head(10))

In [ ]:
port_weather_dfs = []
skipped_ports = 0

for _, row in dest_ports.iterrows():
    port_name = row['destination_port']
    lat, lon  = row['dest_lat'], row['dest_lon']
    safe_name = port_name.replace('/', '_').replace(' ', '_')
    cache_file = WEATHER_CACHE / f'port_{safe_name}.parquet'

    if cache_file.exists():
        df_pw = pd.read_parquet(cache_file)
    else:
        try:
            df_pw = fetch_weather(lat, lon, DATE_START, DATE_END)
            df_pw['destination_port'] = port_name
            df_pw.to_parquet(cache_file, index=False)
            time.sleep(0.15)
        except Exception as exc:
            print(f'  Port weather failed for {port_name}: {exc}')
            skipped_ports += 1
            continue

    port_weather_dfs.append(df_pw)

df_port_weather = pd.concat(port_weather_dfs, ignore_index=True)
df_port_weather['time'] = pd.to_datetime(df_port_weather['time'])

# Rename columns to avoid collisions with vessel-position weather
rename_map = {
    'wind_speed_10m'    : 'port_wind_speed',
    'wind_gusts_10m'    : 'port_wind_gusts',
    'wind_direction_10m': 'port_wind_direction',
    'precipitation'     : 'port_precipitation',
    'weather_code'      : 'port_weather_code',
    'visibility'        : 'port_visibility',
}
df_port_weather = df_port_weather.rename(columns=rename_map)

print(f'Port weather records fetched : {len(df_port_weather):,}')
print(f'Ports skipped                : {skipped_ports}')

In [ ]:
# Merge port weather onto sea pings by destination_port + floored hour
df_sea = df_sea.merge(
    df_port_weather.rename(columns={'time': 'hour'}),
    on=['destination_port', 'hour'],
    how='left',
)

port_wind_coverage = df_sea['port_wind_speed'].notna().mean()
print(f'Port wind coverage : {port_wind_coverage:.1%}  '
      f'({df_sea["port_wind_speed"].notna().sum():,} / {len(df_sea):,} pings)')

display(
    df_sea[['destination_port', 'port_wind_speed', 'port_wind_gusts',
            'port_precipitation', 'port_weather_code', 'port_visibility']]
    .dropna(subset=['port_wind_speed'])
    .head(8)
    .round(2)
)

In [ ]:
# Destination-port weather EDA
# --- Top 15 destination ports by ping count ---
top_ports = df_sea['destination_port'].value_counts().head(15).index

port_stats = (
    df_sea[df_sea['destination_port'].isin(top_ports)]
    .groupby('destination_port')
    .agg(
        ping_count=('sog', 'count'),
        mean_port_wind=('port_wind_speed', 'mean'),
        max_port_gusts=('port_wind_gusts', 'max'),
        total_precip=('port_precipitation', 'sum'),
        mean_remaining_hrs=('remaining_travel_hours', 'mean'),
    )
    .sort_values('mean_port_wind', ascending=True)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Destination-port weather — top 15 busiest ports', fontsize=12)

# Wind speed per port
port_stats['mean_port_wind'].plot.barh(
    ax=axes[0], color='darkorange', alpha=0.8
)
axes[0].set_xlabel('Mean wind speed at port (m/s)')
axes[0].set_title('Mean wind speed')
axes[0].grid(axis='x', alpha=0.3)

# Precipitation vs mean remaining hours
sc = axes[1].scatter(
    port_stats['total_precip'],
    port_stats['mean_remaining_hrs'],
    s=port_stats['ping_count'] / 50,
    c=port_stats['max_port_gusts'],
    cmap='YlOrRd',
    alpha=0.8,
    edgecolors='black',
    linewidths=0.5,
)
plt.colorbar(sc, ax=axes[1], label='Max gust (m/s)')
axes[1].set_xlabel('Total precipitation at port (mm, 9-day window)')
axes[1].set_ylabel('Mean remaining travel hours')
axes[1].set_title('Precipitation vs. remaining hours (bubble = ping count)')
axes[1].grid(alpha=0.3)

for port, row in port_stats.iterrows():
    axes[1].annotate(
        port, (row['total_precip'], row['mean_remaining_hrs']),
        fontsize=6, xytext=(4, 2), textcoords='offset points',
    )

plt.tight_layout()
plt.show()

## 5. Derived weather features

Raw weather variables are useful but domain-specific transformations expose the
physical mechanisms more directly.

| Feature | Formula | Interpretation |
|---|---|---|
| `headwind_component` | `wind_speed × cos(cog − wind_dir)` | >0 headwind slows vessel; <0 tailwind helps |
| `sea_state` | Beaufort scale from wave height | Categorical severity (0–9) |
| `adverse_sea_state` | `wave_height ≥ 2 m OR wind_speed ≥ 10 m/s` | Binary: rough conditions |
| `port_high_wind` | `port_wind_gusts ≥ 15 m/s` | Binary: probable berthing delays |
| `port_precip_flag` | `port_precipitation > 0` | Binary: precipitation event at port |

In [ ]:
def wave_height_to_beaufort(h: pd.Series) -> pd.Series:
    """Map significant wave height (m) to approximate Beaufort sea state."""
    bins   = [-np.inf, 0.1, 0.5, 1.25, 2.5, 4.0, 6.0, 9.0, 14.0, np.inf]
    labels = [0, 1, 2, 3, 4, 5, 6, 7, 8]
    return pd.cut(h, bins=bins, labels=labels, right=True).astype('Int64')


# Headwind component — requires cog and wind_direction_10m
# Angle > 0 → vessel faces into wind (headwind); < 0 → wind pushes from behind
angle_diff = (df_sea['cog'] - df_sea['wind_direction_10m']) % 360
df_sea['headwind_component'] = (
    np.cos(np.radians(angle_diff)) * df_sea['wind_speed_10m']
)  # m/s equivalent headwind (positive = into wind)

# Sea state (Beaufort)
df_sea['sea_state'] = wave_height_to_beaufort(df_sea['wave_height'])

# Binary adverse conditions
df_sea['adverse_sea_state'] = (
    (df_sea['wave_height'] >= 2.0) | (df_sea['wind_speed_10m'] >= 10.0)
).astype('Int8')

df_sea['port_high_wind'] = (df_sea['port_wind_gusts'] >= 15.0).astype('Int8')
df_sea['port_precip_flag'] = (df_sea['port_precipitation'] > 0).astype('Int8')

print('Derived weather feature summary:')
derived_cols = [
    'headwind_component', 'sea_state', 'adverse_sea_state',
    'port_high_wind', 'port_precip_flag',
]
display(df_sea[derived_cols].describe().round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Derived weather features vs vessel behaviour', fontsize=12)

# 1. Headwind component vs SOG scatter (sampled)
df_hw = df_sea.dropna(subset=['headwind_component', 'sog']).sample(
    min(8000, len(df_sea)), random_state=42
)
axes[0].scatter(df_hw['headwind_component'], df_hw['sog'],
                alpha=0.15, s=4, color='steelblue')
# Overlay binned mean
hw_bins = pd.cut(df_hw['headwind_component'], bins=20)
hw_mean = df_hw.groupby(hw_bins, observed=True)['sog'].mean()
hw_mid  = [b.mid for b in hw_mean.index]
axes[0].plot(hw_mid, hw_mean.values, color='red', linewidth=2, label='Binned mean')
axes[0].axvline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
axes[0].set_xlabel('Headwind component (m/s)')
axes[0].set_ylabel('SOG (knots)')
axes[0].set_title('Headwind vs SOG')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# 2. Sea state vs median SOG
ss_sog = (
    df_sea.dropna(subset=['sea_state', 'sog'])
    .groupby('sea_state', observed=True)['sog']
    .agg(['median', 'count'])
    .reset_index()
)
bars = axes[1].bar(
    ss_sog['sea_state'].astype(str),
    ss_sog['median'],
    color='steelblue', alpha=0.8,
)
ax2_twin = axes[1].twinx()
ax2_twin.plot(range(len(ss_sog)), ss_sog['count'], 'o--',
              color='black', markersize=4, linewidth=1, label='Ping count')
ax2_twin.set_ylabel('Ping count (log)', color='black')
ax2_twin.set_yscale('log')
axes[1].set_xlabel('Beaufort sea state')
axes[1].set_ylabel('Median SOG (knots)')
axes[1].set_title('Sea state vs median SOG')
axes[1].grid(alpha=0.3)

# 3. Port high-wind flag vs remaining travel hours
df_port_flag = df_sea.dropna(subset=['port_high_wind', 'remaining_travel_hours'])
for flag, label, colour in [(0, 'Normal port conditions', 'steelblue'),
                             (1, 'High wind at port (≥15 m/s)', 'crimson')]:
    subset = df_port_flag[df_port_flag['port_high_wind'] == flag]['remaining_travel_hours']
    axes[2].hist(subset, bins=40, alpha=0.5, color=colour, label=f'{label} (n={len(subset):,})',
                 density=True)
axes[2].set_xlabel('Remaining travel hours')
axes[2].set_ylabel('Density')
axes[2].set_title('Remaining hours: normal vs high-wind port')
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix: weather features vs SOG and remaining_travel_hours
analysis_cols = [
    # Vessel-position weather
    'wave_height', 'wave_period', 'swell_wave_height',
    'ocean_current_velocity', 'wind_speed_10m', 'wind_gusts_10m',
    'headwind_component',
    # Destination-port weather
    'port_wind_speed', 'port_wind_gusts', 'port_precipitation',
    # Targets
    'sog', 'remaining_travel_hours',
]

df_corr = df_sea[analysis_cols].dropna(how='all')
# Convert Int8 columns to float for correlation
for col in df_corr.columns:
    if hasattr(df_corr[col], 'dtype') and pd.api.types.is_integer_dtype(df_corr[col]):
        df_corr[col] = df_corr[col].astype('float64')

corr_matrix = df_corr.corr()

# Extract correlation with targets only, sorted by absolute value
target_corr = (
    corr_matrix[['sog', 'remaining_travel_hours']]
    .drop(index=['sog', 'remaining_travel_hours'])
    .sort_values('sog', key=abs, ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Pearson correlation: weather features vs target variables', fontsize=12)

for ax, col, colour_pos, colour_neg in zip(
    axes,
    ['sog', 'remaining_travel_hours'],
    ['steelblue', 'steelblue'],
    ['crimson', 'crimson'],
):
    vals = target_corr[col].sort_values()
    colours = [colour_pos if v >= 0 else colour_neg for v in vals]
    ax.barh(range(len(vals)), vals.values, color=colours, alpha=0.8)
    ax.set_yticks(range(len(vals)))
    ax.set_yticklabels(vals.index, fontsize=9)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Pearson r')
    ax.set_title(f'Correlation with {col}')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print('Correlation with SOG and remaining_travel_hours:')
display(target_corr.round(4))

## 6. Candidate feature set for the prediction model

Based on the EDA above, here are the weather features worth adding to `training.ipynb`:

### Vessel-position features
| Feature | Notes |
|---|---|
| `wave_height` | Strongest marine signal; NaN for inland waterways |
| `swell_wave_height` | Separate from wind waves; independent swell decay |
| `headwind_component` | Physical mechanism; requires `cog` (already in AIS) |
| `wind_speed_10m` | Backup when wave data is NaN (inland) |
| `ocean_current_velocity` | Supplementary; coverage patchier than wave data |
| `sea_state` | Ordinal summary; useful as categorical feature |
| `adverse_sea_state` | Binary flag; may help in tree-based models |

### Destination-port features
| Feature | Notes |
|---|---|
| `port_wind_gusts` | Gusts above ~15 m/s typically trigger berthing delays |
| `port_high_wind` | Binary flag derived from gusts |
| `port_precipitation` | Rainfall reduces visibility and slows pilotage |
| `port_precip_flag` | Binary; captures event presence |
| `port_visibility` | Not always available from ERA5; use where present |

### Implementation notes

- **NaN handling:** `HistGradientBoostingRegressor` handles missing values natively — no imputation needed. Inland waterway pings with `NaN` wave data will simply not split on wave features.
- **Lag features:** For port weather, the conditions at *estimated arrival time* (not current time) are more predictive. This requires a forward-looking join on `(destination_port, hour_of_arrival)`.
- **Data volume:** Weather data adds 9 numeric columns per ping (~72 bytes/row) — negligible for the 712k-ping dataset.
- **Production:** In the Kafka predictor, weather can be fetched from the Open-Meteo forecast API (`api.open-meteo.com`) in real-time with the same interface, no API key required.

### Suggested next steps

1. Add the weather fetch step to `voyage_creator.ipynb` and cache enriched voyages.
2. Re-train `training.ipynb` with the new weather columns and compare MAE/RMSE.
3. Investigate forward-looking port weather: join arrival-time forecast instead of current-time observation.
4. Extend to wind-wave direction relative to shipping lanes (compute lane bearing from port coordinates).